# Air Quality Zone Clustering and Pollution Anomaly Detection Using K-Means and DBSCAN
**DWM Mini-Project** | K-Means + Elbow Method + DBSCAN | Dataset: *Air Quality Data in India* (Rohan Rao, Kaggle)

### Abstract
Air-quality records contain several pollutant measurements per city and day, with missing values, different scales and skewed distributions. This notebook builds an **unsupervised** workflow that (1) groups *cities* with similar pollutant profiles using **K-Means** (K chosen with the **Elbow Method**), (2) groups *daily city observations* with K-Means, and (3) uses **DBSCAN** to find density-based groups and observations that fall outside them (noise points). Results are interpreted with cluster profiles and PCA plots.

> **Integrity note:** this notebook is shipped *without outputs*. Every number (dataset size, K, scores, counts) is printed when **you** run it on your data. Clusters are exploratory groupings, **not** official air-quality categories, and DBSCAN noise points are only *potential* anomalies for further investigation.

## 1. Problem Statement and Objectives
**Problem statement.** Air-quality monitoring datasets contain multiple pollutant measurements across cities and dates, often with missing values, different measurement scales, skewed distributions, and complex patterns that are difficult to summarize using simple averages alone. This project develops an unsupervised clustering workflow to discover similarities in city-level pollutant profiles and daily city-level observations. K-Means clustering is applied, with the Elbow Method used as a heuristic to select a reasonable number of clusters. DBSCAN is then used to identify density-based groups and observations that fall outside these groups. Cluster profiles and visualizations are used to interpret the patterns. The system supports exploratory analysis only; cluster membership and DBSCAN noise labels are not official air-quality classifications or proof of hazardous conditions.

**Objectives**
1. Load and inspect the CSV dataset.
2. Quantify missingness and validate available features.
3. Clean and preprocess numeric pollutant attributes.
4. Aggregate pollutant readings by city for city-level analysis.
5. Use an Elbow curve to help select K for K-Means.
6. Run K-Means on city-level profiles and daily observations.
7. Run DBSCAN on a documented sample (or full data only if computationally appropriate).
8. Visualize clusters using PCA and report explained variance.
9. Export cluster assignments, profiles and potential-noise observations to CSV.
10. Compare the algorithms and discuss limitations.

## 2. Dataset Source and Setup
* **Dataset:** [Air Quality Data in India (2015-2020) - Rohan Rao, Kaggle](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)
* **File used:** `city_day.csv` (daily, city-level readings).
* **Place it at:** `data/raw/city_day.csv` (relative to the project folder). The notebook does **not** download it automatically.
* Start Jupyter from the project root so relative paths work (`jupyter lab`).

## 3. Imports and Configuration
All tunable settings (column names, K overrides, DBSCAN `eps` / `min_samples`, sample sizes, random seed) are in the configuration cell below. `CITY_K`, `DAILY_K` and `DBSCAN_EPS` are `None` by default, meaning "use the heuristic suggestion"; set a number to override.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

# --- Notebook-specific settings -------------------------------------------------
PROJECT_ROOT = Path.cwd()                      # start Jupyter from the project folder
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent         # in case the notebook was opened from a sub-folder
SHOW_PLOTS = True                              # show figures in the notebook as well as saving them
print("Project root:", PROJECT_ROOT)

: 

In [ ]:
# =============================================================================
# CONFIGURATION - every tunable setting lives here. Change values and re-run.
# =============================================================================
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "city_day.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATASET_URL = "https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india"

# Column names (the code checks which of these really exist in the CSV)
CITY_COL, DATE_COL, AQI_COL, BUCKET_COL = "City", "Date", "AQI", "AQI_Bucket"
POLLUTANT_CANDIDATES = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
                        "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]

RANDOM_STATE = 42            # fixed seed -> reproducible results
MIN_FEATURE_COVERAGE = 0.30  # keep a pollutant only if >= 30% of its values are usable
MIN_FEATURES_PER_ROW = 3     # keep a daily row only if it has >= 3 measured pollutants

# --- K-Means / Elbow ---------------------------------------------------------
KMEANS_N_INIT = 10           # number of random restarts for each K-Means fit
CITY_K = None                # None = use the elbow suggestion; or set an int (e.g. 3) to override
CITY_K_MAX = 8               # elbow curve tests K = 2 ... min(8, n_cities - 1)
DAILY_K = None               # None = use the elbow suggestion; or set an int (e.g. 4) to override
DAILY_K_MAX = 10             # elbow curve tests K = 2 ... DAILY_K_MAX

# --- Silhouette / plotting ---------------------------------------------------
SILHOUETTE_SAMPLE_SIZE = 10000   # silhouette on large data is scored on a reproducible sample
PLOT_SAMPLE_SIZE = 15000         # max points drawn in the daily PCA scatter (speed/readability)

# --- DBSCAN ------------------------------------------------------------------
DBSCAN_MAX_SAMPLES = 6000    # DBSCAN is run on at most this many daily observations
DBSCAN_MIN_SAMPLES = 10      # min_samples (the point itself counts toward this number)
DBSCAN_EPS = None            # None = starting value from the k-distance plot; or set a float (e.g. 1.2)

: 

In [ ]:
# =============================================================================
# GENERAL HELPERS
# =============================================================================
def save_figure(fig, filename):
    """Save a figure as a high-resolution PNG in outputs/, show it if allowed, then close it."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    print(f"  saved figure -> outputs/{filename}")


def save_csv(df, filename, index=False):
    """Save a DataFrame as CSV in outputs/."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_DIR / filename, index=index)
    print(f"  saved table  -> outputs/{filename}  ({len(df)} rows)")


def to_builtin(obj):
    """Convert numpy types so the summary can be written as JSON."""
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    return obj

## 4. Dataset Loading and Exploration
We load the CSV, strip stray spaces from column names, parse `Date` with `errors="coerce"` (bad dates become `NaT`), and print shape, columns, dtypes, number of cities, date range and missing values. If the file is missing, a clear message tells you where to put it.

In [ ]:
# =============================================================================
# 1. DATA LOADING AND EXPLORATION
# =============================================================================
def load_dataset(csv_path):
    """Read the CSV. Stops with a clear message if the file is missing."""
    if not Path(csv_path).exists():
        raise FileNotFoundError(
            "\nDataset not found.\n"
            f"  Expected location : {csv_path}\n"
            f"  Download from     : {DATASET_URL}\n"
            "  Use the file 'city_day.csv' and place it at data/raw/city_day.csv\n"
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()          # remove accidental spaces in headers
    if CITY_COL not in df.columns:
        raise ValueError(f"Column '{CITY_COL}' not found. Available columns: {list(df.columns)}")
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    return df


def inspect_dataset(df):
    """Print basic facts about the loaded data (nothing is hard-coded)."""
    print("Shape (rows, columns):", df.shape)
    print("\nColumn names:", list(df.columns))
    print("\nData types:\n", df.dtypes)
    print("\nFirst 5 rows:\n", df.head())
    print("\nNumber of distinct cities:", df[CITY_COL].nunique())
    if DATE_COL in df.columns and df[DATE_COL].notna().any():
        print("Date range:", df[DATE_COL].min().date(), "to", df[DATE_COL].max().date())
        print("Rows with unparseable dates:", int(df[DATE_COL].isna().sum()))
    print("\nMissing values per column:\n", df.isna().sum())

: 

In [ ]:
try:
    df = load_dataset(DATA_PATH)
except FileNotFoundError as err:
    raise SystemExit(str(err))   # stops the notebook with a clear message
inspect_dataset(df)

## 5. Missing-Value Analysis
Real sensor data has gaps. Here we measure the percentage of missing values per pollutant *before* any cleaning. Pollutants with very little data (below `MIN_FEATURE_COVERAGE`) will be dropped in the next step rather than filled with mostly-invented numbers.

In [ ]:
# =============================================================================
# 2. MISSING-VALUE ANALYSIS
# =============================================================================
def missing_percent_table(df, columns):
    """Percentage of missing values for the given columns (after numeric conversion)."""
    numeric = df[columns].apply(pd.to_numeric, errors="coerce")
    return (numeric.isna().mean() * 100).round(2)


def plot_missing_values(raw_pct, clean_pct, filename="missing_values.png"):
    """Bar chart comparing missing % before and after cleaning."""
    table = pd.DataFrame({
        "Before cleaning": raw_pct,
        "After cleaning (negatives -> NaN, sparse rows dropped)": clean_pct,
    })
    table = table.rename_axis("Pollutant").reset_index()
    long = table.melt(id_vars="Pollutant", var_name="Stage", value_name="Missing (%)")
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.barplot(data=long, x="Pollutant", y="Missing (%)", hue="Stage", ax=ax)
    ax.set_title("Missing values per pollutant")
    ax.set_ylabel("Missing values (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Stage")
    save_figure(fig, filename)

In [ ]:
pollutant_candidates_present = [c for c in POLLUTANT_CANDIDATES if c in df.columns]
raw_missing = missing_percent_table(df, pollutant_candidates_present)
print(raw_missing)

## 6. Data Validation and Preprocessing
Steps, in order:
1. **Select pollutants** that exist and have enough usable data. `AQI`, `AQI_Bucket`, `City` and `Date` are *metadata*: they are kept for interpretation but are **never** used as clustering features (AQI is itself computed from pollutants, so including it would double-count them and leak the official label into the clusters).
2. **Clean:** convert to numbers; set negative concentrations to missing (they are physically invalid and `log1p` would break); drop rows with too few measurements; drop duplicate City+Date rows.
3. **Median imputation** for remaining gaps (robust to outliers; note that imputation pulls values toward the middle, so it affects interpretation).
4. **`log1p` transform** - pollutant concentrations are right-skewed and non-negative; the log compresses extreme spikes.
5. **`StandardScaler`** - K-Means and DBSCAN are distance-based, so a pollutant measured in hundreds (PM10) would otherwise dominate one measured in single digits (Benzene).

In [ ]:
# =============================================================================
# 3. VALIDATION AND PREPROCESSING
# =============================================================================
def select_pollutant_columns(df):
    """Pick pollutant columns that exist AND contain enough usable numeric data."""
    present = [c for c in POLLUTANT_CANDIDATES if c in df.columns]
    if not present:
        raise ValueError("None of the expected pollutant columns were found. "
                         f"Expected some of {POLLUTANT_CANDIDATES}; found {list(df.columns)}")
    numeric = df[present].apply(pd.to_numeric, errors="coerce")
    coverage = numeric.notna().mean()
    selected = [c for c in present if coverage[c] >= MIN_FEATURE_COVERAGE]
    dropped = [c for c in present if c not in selected]
    absent = [c for c in POLLUTANT_CANDIDATES if c not in df.columns]
    print("Pollutant columns present  :", present)
    print("Pollutant columns absent   :", absent)
    print(f"Dropped (coverage < {MIN_FEATURE_COVERAGE:.0%}):", dropped)
    print("Pollutant columns SELECTED :", selected)
    if len(selected) < 2:
        raise ValueError("Fewer than 2 usable pollutant columns - cannot cluster or draw PCA.")
    return selected, present


def clean_pollutants(df, pollutants):
    """
    Cleaning policy (documented):
      1. Convert pollutant columns to numbers (bad text -> NaN).
      2. Negative concentrations are physically invalid -> set to NaN (NOT passed to log1p).
      3. Drop rows with no City, and rows with fewer than MIN_FEATURES_PER_ROW measured pollutants.
      4. Drop exact duplicate City+Date rows (if Date exists).
    Metadata (City, Date, AQI, AQI_Bucket) is kept in the table but is never a clustering feature.
    """
    work = df.copy()
    work[pollutants] = work[pollutants].apply(pd.to_numeric, errors="coerce")
    if AQI_COL in work.columns:
        work[AQI_COL] = pd.to_numeric(work[AQI_COL], errors="coerce")

    neg_mask = work[pollutants] < 0
    n_negative = int(neg_mask.sum().sum())
    work[pollutants] = work[pollutants].mask(neg_mask)

    n_start = len(work)
    work = work[work[CITY_COL].notna()]
    n_enough = work[pollutants].notna().sum(axis=1) >= min(MIN_FEATURES_PER_ROW, len(pollutants))
    work = work[n_enough]
    n_duplicates = 0
    if DATE_COL in work.columns:
        before = len(work)
        work = work.drop_duplicates(subset=[CITY_COL, DATE_COL])
        n_duplicates = before - len(work)
    work = work.reset_index(drop=True)

    stats = {"rows_before_cleaning": n_start, "rows_after_cleaning": len(work),
             "rows_removed": n_start - len(work), "negative_values_set_to_nan": n_negative,
             "duplicate_city_date_rows_removed": n_duplicates}
    print("Cleaning summary:", stats)
    return work, stats


def transform_features(df, columns):
    """
    Median imputation -> log1p -> StandardScaler.
    Returns (scaled matrix, list of columns actually used, fitted scaler).
    NOTE: this is an unsupervised analysis on the full dataset (no train/test split), so
    the imputer and scaler are fitted once on the data being clustered; the same fitted
    scaler is then reused for every later step (no refitting on subsets -> no inconsistencies).
    """
    used = [c for c in columns if df[c].notna().any()]       # drop all-empty features
    imputer = SimpleImputer(strategy="median")
    imputed = imputer.fit_transform(df[used])
    if (imputed < 0).any():
        raise ValueError("Negative values reached log1p - cleaning step failed.")
    logged = np.log1p(imputed)                                # tames right-skewed concentrations
    scaler = StandardScaler()
    scaled = scaler.fit_transform(logged)                     # mean 0, std 1 per feature
    return scaled, used, scaler

In [ ]:
pollutants, present = select_pollutant_columns(df)
df_clean, clean_stats = clean_pollutants(df, pollutants)
clean_missing = missing_percent_table(df_clean, pollutants).reindex(present)
print("\nMissing % BEFORE cleaning:\n", raw_missing)
print("\nMissing % AFTER cleaning (before imputation):\n", clean_missing)
print("\nAfter median imputation: 0% missing in every used feature (by construction).")
plot_missing_values(raw_missing, clean_missing)
meta_cols = [c for c in [CITY_COL, DATE_COL, AQI_COL, BUCKET_COL] if c in df_clean.columns]

## 7. City Aggregation and City-Level Elbow Method
**Aggregation choice:** we use the **median** of each pollutant per city because daily pollution is right-skewed with spikes; the median is a more robust "typical level" than the mean. Cities with all-missing features are dropped; remaining gaps are median-imputed across cities. City features are scaled **separately** from daily features.

**Elbow Method:** K-Means is run for K = 2 ... min(8, n_cities - 1). *Inertia* (within-cluster sum of squares) always decreases as K grows; we look for the "elbow" where extra clusters stop giving big improvements. This is a **heuristic, not a universally correct K** - a helper suggests a K, and you can override it with `CITY_K`.

In [ ]:
# =============================================================================
# 4. K-MEANS HELPERS (ELBOW, SILHOUETTE, PROFILES)
# =============================================================================
def valid_k_values(n_samples, k_max):
    """K must be >= 2 and smaller than the number of samples."""
    upper = min(k_max, n_samples - 1)
    if upper < 2:
        raise ValueError(f"Not enough samples ({n_samples}) to try K >= 2.")
    return list(range(2, upper + 1))


def compute_inertias(X, k_values):
    """Inertia = sum of squared distances of points to their nearest centroid (WCSS)."""
    inertias = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
        model.fit(X)
        inertias.append(float(model.inertia_))
    return inertias


def suggest_k_elbow(k_values, inertias):
    """
    Heuristic 'elbow': the K whose point is farthest from the straight line joining the first
    and last points of the (normalised) curve. It is only a SUGGESTION - always look at the plot.
    """
    x = np.array(k_values, dtype=float)
    y = np.array(inertias, dtype=float)
    if len(x) < 3 or np.ptp(y) == 0:
        return int(x[0])
    xn = (x - x.min()) / np.ptp(x)
    yn = (y - y.min()) / np.ptp(y)
    dx, dy = xn[-1] - xn[0], yn[-1] - yn[0]
    distance = np.abs(dy * (xn - xn[0]) - dx * (yn - yn[0])) / np.hypot(dx, dy)
    return int(x[int(np.argmax(distance))])


def plot_elbow(k_values, inertias, suggested_k, title, filename):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(k_values, inertias, marker="o")
    ax.axvline(suggested_k, linestyle="--", label=f"Heuristic elbow suggestion: K = {suggested_k}")
    ax.set_xlabel("Number of clusters (K)")
    ax.set_ylabel("Inertia (within-cluster sum of squares)")
    ax.set_title(title)
    ax.set_xticks(k_values)
    ax.legend()
    save_figure(fig, filename)


def choose_k(override, k_values, inertias, name):
    """Use the user's override if given (and valid); otherwise the elbow suggestion."""
    suggested = suggest_k_elbow(k_values, inertias)
    if override is not None:
        if override < 2 or override > max(k_values):
            raise ValueError(f"{name} override ({override}) must be between 2 and {max(k_values)}.")
        print(f"{name}: elbow suggested K={suggested}, but manual override K={override} is used.")
        return int(override), suggested
    print(f"{name}: using elbow-suggested K={suggested} (set {name} in the config to override).")
    return suggested, suggested


def fit_kmeans(X, k):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
    labels = model.fit_predict(X)
    return model, labels


def safe_silhouette(X, labels):
    """
    Silhouette score in [-1, 1] (higher = tighter, better-separated clusters).
    Returned only when valid: at least 2 clusters and fewer clusters than observations.
    Large data are scored on a reproducible random sample.
    """
    n, n_clusters = len(labels), len(np.unique(labels))
    if n_clusters < 2 or n_clusters >= n:
        print("Silhouette skipped: needs >= 2 clusters and fewer clusters than observations.")
        return None
    sample = SILHOUETTE_SAMPLE_SIZE if n > SILHOUETTE_SAMPLE_SIZE else None
    try:
        return float(silhouette_score(X, labels, sample_size=sample, random_state=RANDOM_STATE))
    except ValueError as err:
        print("Silhouette could not be computed:", err)
        return None


def make_profile(df, columns, label_col, list_col=None):
    """Cluster profile in ORIGINAL units: median of each pollutant, plus size (and AQI info)."""
    grouped = df.groupby(label_col)
    profile = grouped[columns].median().round(2)
    profile.insert(0, "n_members", grouped.size())
    if AQI_COL in df.columns:
        profile["AQI_median"] = grouped[AQI_COL].median().round(1)
    if BUCKET_COL in df.columns:
        profile["most_common_AQI_Bucket"] = grouped[BUCKET_COL].agg(
            lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
    if list_col is not None:
        profile["members"] = grouped[list_col].agg(lambda s: ", ".join(sorted(map(str, s))))
    return profile.reset_index()

In [ ]:
city_df = df_clean.groupby(CITY_COL)[pollutants].median()
city_df = city_df.dropna(axis=1, how="all").dropna(axis=0, how="all")
city_df["n_days"] = df_clean.groupby(CITY_COL).size().reindex(city_df.index)
if AQI_COL in df_clean.columns:
    city_df[AQI_COL] = df_clean.groupby(CITY_COL)[AQI_COL].median().reindex(city_df.index)
city_df = city_df.reset_index()
city_features = [c for c in pollutants if c in city_df.columns]
X_city, city_features, _ = transform_features(city_df, city_features)
print("Cities clustered:", len(city_df), "| features:", city_features)

city_ks = valid_k_values(len(city_df), CITY_K_MAX)
city_inertias = compute_inertias(X_city, city_ks)
city_k, city_suggested = choose_k(CITY_K, city_ks, city_inertias, "CITY_K")
plot_elbow(city_ks, city_inertias, city_suggested, "Elbow Method - city-level profiles", "city_elbow.png")

## 8. City-Level K-Means Clustering
K-Means (`random_state=42`, `n_init=10`) groups cities with similar standardised pollutant profiles.

In [ ]:
_, city_labels = fit_kmeans(X_city, city_k)
city_df["city_cluster"] = city_labels
print(city_df[[CITY_COL, "city_cluster"]].sort_values("city_cluster").to_string(index=False))

## 9. City Cluster Profiles and Interpretation Prompts
Profiles are shown in **original units** (median per cluster), so you can read them directly.

**Interpretation prompts (answer these in your report using the printed numbers):**
* Which cluster has the highest PM2.5 / PM10? Which has the lowest overall pollution?
* Do the clusters differ mainly in particulate matter, gases (NO2, SO2, CO) or both?
* Do cities within a cluster share a region or climate? (Do not assume - check the member list.)
* How does the cluster's median AQI line up with its pollutant levels?

In [ ]:
city_profile = make_profile(city_df, city_features, "city_cluster", list_col=CITY_COL)
city_profile

## 10. Daily-Observation Elbow Method
Same idea at a finer level: each row (one city on one day) is a point. K is tested for 2 ... `DAILY_K_MAX`. Change `DAILY_K` in the configuration to override the suggestion.

In [ ]:
X_daily, daily_features, _ = transform_features(df_clean, pollutants)
print("Daily observations:", X_daily.shape[0], "| features:", daily_features)
daily_ks = valid_k_values(len(X_daily), DAILY_K_MAX)
daily_inertias = compute_inertias(X_daily, daily_ks)
daily_k, daily_suggested = choose_k(DAILY_K, daily_ks, daily_inertias, "DAILY_K")
plot_elbow(daily_ks, daily_inertias, daily_suggested, "Elbow Method - daily observations", "daily_elbow.png")

## 11. Daily-Observation K-Means
Labels are stored next to the original metadata and **original-unit** pollutant values (not the scaled ones) so results stay interpretable.

In [ ]:
_, daily_labels = fit_kmeans(X_daily, daily_k)
daily_out = df_clean[meta_cols + pollutants].copy()
daily_out["kmeans_cluster"] = daily_labels
daily_profile = make_profile(daily_out, pollutants, "kmeans_cluster")
daily_profile

## 12. Silhouette Score (with validation)
The silhouette score ranges from -1 to 1 and compares each point's closeness to its own cluster with its closeness to the nearest other cluster. It is computed **only** if there are >= 2 clusters and fewer clusters than observations; on large data it uses a reproducible sample.

**Limitations:** silhouette favours compact, round clusters, so it can undervalue useful clusters of other shapes; it does not know whether a cluster is *meaningful* for air-quality interpretation; and it is a single number - a higher score is not the only measure of usefulness. Always read it together with the cluster profiles.

In [ ]:
city_sil = safe_silhouette(X_city, city_labels)
daily_sil = safe_silhouette(X_daily, daily_labels)
print("City-level silhouette score :", city_sil)
print("Daily-level silhouette score:", daily_sil, "(sample-based)" if len(X_daily) > SILHOUETTE_SAMPLE_SIZE else "")

## 13. DBSCAN - Parameters Explained
DBSCAN groups points by **density** instead of distance to a centroid, so the number of clusters is *not* chosen in advance.

* **`eps`** - the neighbourhood radius. Two points are neighbours if their distance is <= `eps`.
* **`min_samples`** - how many points (including the point itself) must lie within `eps` for a point to be a **core point**.
* **Core point** - has >= `min_samples` points within `eps`.
* **Border point** - not a core point, but lies within `eps` of a core point; it joins that cluster.
* **Noise point** (label `-1`) - neither core nor border. These are only *potential* unusual observations; they may be rare but legitimate readings, sensor faults, or simply sparse regions.

DBSCAN can return **all noise** (`eps` too small), **one giant cluster** (`eps` too large), or something useful - depending on the feature space and parameters. There is no universally optimal `eps`.

**Sampling:** to keep DBSCAN fast, it runs on a reproducible random sample of at most `DBSCAN_MAX_SAMPLES` daily observations. Results are therefore **sample-based** and do not describe every row in the dataset.

## 14. k-Distance Plot and Configurable `eps` / `min_samples`
For each sampled point we compute the distance to its `min_samples`-th nearest neighbour and sort these distances. The "knee" of the curve is a reasonable *starting* value for `eps`: before the knee points are in dense areas, after it they are increasingly isolated. Edit `DBSCAN_EPS` / `DBSCAN_MIN_SAMPLES` in the configuration cell to experiment.

In [ ]:
# =============================================================================
# 6. DBSCAN HELPERS
# =============================================================================
def k_distances(X, k):
    """Distance from every point to its k-th nearest neighbour (the point itself counts as 1st)."""
    nn = NearestNeighbors(n_neighbors=min(k, len(X))).fit(X)
    distances, _ = nn.kneighbors(X)
    return distances[:, -1]


def suggest_eps(sorted_kdist):
    """Starting value for eps: the bend ('knee') of the sorted k-distance curve."""
    y = np.asarray(sorted_kdist, dtype=float)
    if len(y) < 3 or np.ptp(y) == 0:
        return float(np.median(y))
    x = np.linspace(0, 1, len(y))
    yn = (y - y.min()) / np.ptp(y)
    distance = np.abs((yn[-1] - yn[0]) * (x - x[0]) - (x[-1] - x[0]) * (yn - yn[0]))
    return float(y[int(np.argmax(distance))])


def plot_k_distance(sorted_kdist, k, eps, filename="dbscan_k_distance.png"):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.plot(sorted_kdist)
    ax.axhline(eps, linestyle="--", label=f"eps used = {eps:.3f}")
    ax.set_xlabel("Observations (sorted by distance)")
    ax.set_ylabel(f"Distance to the {k}-th nearest neighbour")
    ax.set_title("DBSCAN k-distance plot (look for the bend to choose eps)")
    ax.legend()
    save_figure(fig, filename)


def describe_extreme_features(X_scaled, feature_names):
    """For each row: which feature is the most extreme (largest |z|) and in which direction."""
    j = np.abs(X_scaled).argmax(axis=1)
    values = X_scaled[np.arange(len(X_scaled)), j]
    names = np.array(feature_names)[j]
    return names, np.where(values > 0, "high", "low")

In [ ]:
n_all = len(X_daily)
sample_size = min(DBSCAN_MAX_SAMPLES, n_all)
sample_idx = np.sort(np.random.default_rng(RANDOM_STATE).choice(n_all, sample_size, replace=False))
X_sample = X_daily[sample_idx]
print(f"DBSCAN sample: {sample_size} of {n_all} observations (sample-based, reproducible).")

if sample_size < 3:
    raise SystemExit("Sample too small for DBSCAN.")
min_samples = min(DBSCAN_MIN_SAMPLES, sample_size)      # prevents crash when sample < min_samples
kdist = k_distances(X_sample, min_samples)
sorted_kdist = np.sort(kdist)
eps_suggested = suggest_eps(sorted_kdist)
eps = float(DBSCAN_EPS) if DBSCAN_EPS is not None else eps_suggested
if eps <= 0:
    raise ValueError("eps must be > 0. Set DBSCAN_EPS in the configuration cell.")
print(f"min_samples = {min_samples} | eps = {eps:.4f} "
      f"({'manual' if DBSCAN_EPS is not None else 'from the k-distance bend - a starting point only'})")
plot_k_distance(sorted_kdist, min_samples, eps)

## 15. DBSCAN Fit and Noise-Point Analysis
Noise count excludes the label `-1` from the cluster count. The warnings below flag the two classic failure modes.

In [ ]:
db_labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_sample)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int((db_labels == -1).sum())
print(f"DBSCAN clusters (excluding noise): {n_clusters}")
print(f"Noise points (label -1): {n_noise}  ({n_noise / sample_size:.1%} of the sample)")
if n_clusters == 0:
    print("WARNING: every point is noise -> eps is probably too small (or min_samples too large).")
elif n_clusters == 1 and n_noise < 0.01 * sample_size:
    print("NOTE: one big cluster and almost no noise -> eps may be too large.")

sample_out = df_clean.iloc[sample_idx][meta_cols + pollutants].copy()
sample_out["dbscan_label"] = db_labels
sample_out["is_noise"] = db_labels == -1
sample_out["k_distance"] = kdist.round(4)
extreme_name, extreme_dir = describe_extreme_features(X_sample, daily_features)
sample_out["most_extreme_feature"] = extreme_name
sample_out["extreme_direction"] = extreme_dir
potential = sample_out[sample_out["is_noise"]].sort_values("k_distance", ascending=False)
potential.head(10)

**Reading the noise table:** `k_distance` is how far the point is from its neighbours (bigger = more isolated). `most_extreme_feature` / `extreme_direction` show which standardised (log-scale) pollutant deviates most - a hint for investigation, not an explanation. *These are potential anomalies, not confirmed pollution emergencies or faulty sensors.*

## 16. PCA Visualisations with Explained Variance
PCA compresses the standardised features to two axes so we can draw them. The two axes keep only part of the information (shown in the title); overlap or separation in this picture is **not proof** of how separable the clusters are in the full feature space.

In [ ]:
# =============================================================================
# 5. PCA VISUALISATION
# =============================================================================
def plot_pca_clusters(X, labels, title, filename, annotate=None, max_points=None, noise_label=None):
    """
    Project the standardised features onto 2 principal components and colour by cluster.
    PCA is used ONLY for drawing - it does not prove that clusters are truly separable.
    Returns the explained-variance ratio of the two components.
    """
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X)
    ratio = pca.explained_variance_ratio_

    index = np.arange(len(coords))
    if max_points is not None and len(index) > max_points:
        index = np.sort(np.random.default_rng(RANDOM_STATE).choice(index, max_points, replace=False))

    fig, ax = plt.subplots(figsize=(9, 6.5))
    for lab in np.unique(labels):
        mask = labels[index] == lab
        pts = coords[index][mask]
        if noise_label is not None and lab == noise_label:
            ax.scatter(pts[:, 0], pts[:, 1], c="black", marker="x", s=22, label=f"Noise (n={mask.sum()})")
        else:
            ax.scatter(pts[:, 0], pts[:, 1], s=28 if annotate is not None else 10, alpha=0.7,
                       label=f"Cluster {lab} (n={mask.sum()})")
    if annotate is not None:                       # label each point (used for the city plot)
        for (px, py), name in zip(coords, annotate):
            ax.annotate(str(name), (px, py), fontsize=7, alpha=0.8, xytext=(3, 3),
                        textcoords="offset points")
    ax.set_xlabel(f"PC1 ({ratio[0]:.1%} of variance)")
    ax.set_ylabel(f"PC2 ({ratio[1]:.1%} of variance)")
    ax.set_title(f"{title}\nPCA explained variance (PC1+PC2) = {ratio.sum():.1%}")
    ax.legend(fontsize=8, loc="best")
    save_figure(fig, filename)
    return [float(r) for r in ratio]

In [ ]:
city_ratio = plot_pca_clusters(X_city, city_labels, f"City-level K-Means (K={city_k})",
                               "city_kmeans_pca.png", annotate=city_df[CITY_COL])
daily_ratio = plot_pca_clusters(X_daily, daily_labels, f"Daily observations K-Means (K={daily_k})",
                                "daily_kmeans_pca.png", max_points=PLOT_SAMPLE_SIZE)
db_ratio = plot_pca_clusters(X_sample, db_labels, "DBSCAN on sampled daily observations",
                             "dbscan_pca.png", noise_label=-1)
print("Explained variance ratio (PC1, PC2):")
print("  city  :", [round(r, 3) for r in city_ratio])
print("  daily :", [round(r, 3) for r in daily_ratio])
print("  dbscan:", [round(r, 3) for r in db_ratio])

## 17. Export Results
All files go to the `outputs/` folder (created automatically). Daily and DBSCAN files keep metadata and **original-unit** pollutant values.

In [ ]:
import json
save_csv(city_df.sort_values("city_cluster"), "city_cluster_results.csv")
save_csv(city_profile, "city_cluster_profiles.csv")
save_csv(daily_out, "daily_kmeans_results.csv")
save_csv(daily_profile, "daily_cluster_profiles.csv")
save_csv(sample_out, "dbscan_results_sample.csv")
save_csv(potential, "potential_pollution_anomalies.csv")

summary = {
    "raw_shape": list(df.shape), "pollutants_used": pollutants, "cleaning": clean_stats,
    "city_level": {"n_cities": len(city_df), "elbow_suggested_k": city_suggested, "k_used": city_k,
                   "silhouette": city_sil, "pca_explained_variance": city_ratio},
    "daily_level": {"n_observations": int(X_daily.shape[0]), "elbow_suggested_k": daily_suggested,
                    "k_used": daily_k, "silhouette": daily_sil, "pca_explained_variance": daily_ratio},
    "dbscan": {"sample_size": sample_size, "total_observations": n_all, "eps": eps,
               "min_samples": min_samples, "n_clusters_excluding_noise": n_clusters, "n_noise": n_noise},
}
with open(OUTPUT_DIR / "run_summary.json", "w") as f:
    json.dump(to_builtin(summary), f, indent=2)
print("Saved outputs/run_summary.json - use these real numbers in your report.")

## 18. K-Means vs DBSCAN Comparison
| Aspect | K-Means | DBSCAN |
|---|---|---|
| Idea | Partition points around K centroids | Grow clusters from dense regions |
| Number of clusters | Must be chosen (Elbow, silhouette, domain sense) | Found automatically from `eps` and `min_samples` |
| Cluster shape | Tends toward compact, roughly spherical | Arbitrary shapes |
| Outliers | Every point is forced into a cluster | Low-density points get label `-1` (noise) |
| Main parameters | `K`, `n_init`, `random_state` | `eps`, `min_samples` |
| Sensitivity | Sensitive to scale, outliers, initialisation | Very sensitive to `eps`; struggles with varying densities |
| Cost | Fast, scales well to the full dataset | Neighbour searches; here run on a sample |
| Best use here | Profiling typical pollution patterns / city groups | Flagging unusual observations for review |

Use your own printed results to comment on how the two methods differed on this dataset.

## 19. Conclusion Template (fill in from YOUR printed outputs)
Write the conclusion only after running the notebook. Suggested structure - replace each `<...>` with a real value from `outputs/run_summary.json` and the profile tables:

1. The cleaned data contained `<rows>` daily observations from `<n_cities>` cities using `<pollutants>`.
2. City-level K-Means used K = `<city_k>` (elbow suggested `<city_suggested>`); the clusters were `<describe highest / lowest pollution groups with real city names>`. Silhouette = `<value>` (interpret cautiously).
3. Daily K-Means used K = `<daily_k>`; cluster profiles showed `<describe>`.
4. DBSCAN (sample of `<sample_size>`, eps = `<eps>`, min_samples = `<min_samples>`) found `<n_clusters>` clusters and `<n_noise>` noise points (`<percent>`%).
5. Limitations: imputation, log transform, sampling, heuristic K and eps, no ground truth, PCA captures only `<x>`% of variance.
6. Cluster IDs and noise labels are exploratory, not official classifications or proof of hazardous conditions.

## 20. Viva Questions and Answers
**1. What type of learning is this?** Unsupervised - there are no target labels; we look for structure in the data.

**2. Why standardise the features?** K-Means and DBSCAN use Euclidean distance; without scaling, pollutants with large numeric ranges dominate the distance.

**3. Why `log1p`?** Concentrations are non-negative and right-skewed; `log(1+x)` compresses spikes and is defined at 0. It needs x > -1, so negatives are removed first.

**4. Why median imputation?** It is robust to outliers and keeps the data size, but it shrinks variation and affects interpretation.

**5. What is inertia?** The sum of squared distances from each point to its assigned centroid (within-cluster sum of squares).

**6. What is the Elbow Method and its limitation?** Plot inertia vs K and pick where improvement slows. It is subjective and sometimes has no clear elbow.

**7. Why `random_state` and `n_init`?** K-Means starts from random centroids; a fixed seed makes results reproducible and several restarts reduce poor local minima.

**8. What are `eps` and `min_samples`?** `eps` is the neighbourhood radius; `min_samples` is the number of points needed within `eps` to form a dense region (core point).

**9. Core vs border vs noise points?** Core: dense neighbourhood. Border: within `eps` of a core point but not dense itself. Noise: neither.

**10. How did you choose `eps`?** From the bend of the sorted k-distance plot as a starting point, then inspected the resulting clusters/noise; it is not universally optimal.

**11. Are DBSCAN noise points confirmed anomalies?** No - they are potential unusual observations needing domain verification.

**12. Why is PCA used?** Only to visualise high-dimensional data in 2D; it does not prove separability and loses some variance.

**13. What does the silhouette score tell us and what are its limits?** How well points fit their own cluster versus others (-1 to 1). It favours compact clusters and is not the only measure of usefulness.

**14. Why not use AQI or AQI_Bucket as features?** AQI is derived from pollutants, so it would duplicate information and bias clusters; they are kept only for interpretation.

**15. Why was DBSCAN run on a sample?** To limit computation; results are sample-based and must not be described as covering the full dataset.

**16. K-Means vs DBSCAN in one line?** K-Means partitions all points into K compact groups; DBSCAN finds dense groups of any shape and marks sparse points as noise.